In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, mean_absolute_error, r2_score)

# Load the dataset (update the path if your file is located elsewhere)
df = pd.read_csv("C:/Users/woons/Documents/Python Projects/capstone_part2/part3_machine learning/data/features_traffic_proxy.csv")

# ==========================================
# 1. PREPARE COMMON FEATURE SET
# ==========================================

# Create Holiday Flag (Assuming 'None' represents typical non-holiday days)
df['is_holiday'] = (df['holiday'] != 'None').astype(int)

# Cyclical encoding for day of the week (0-6)
# (hour_sin and hour_cos were already created in Part 2)
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Define the common feature columns explicitly 
# (Excludes target columns, raw text, and leakage columns like congestion_category)
features = [
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend', # Time & Cyclical
    'temp_scaled', 'rain_1h_scaled', 'clouds_all_scaled',       # Weather (continuous)
    'weather_main_code', 'is_rainy', 'is_snowy',                # Weather (categorical/flags)
    'is_holiday'                                                # Holiday flag
]

X = df[features]


# ==========================================
# 2. CLASSIFICATION TASK (Target: high_risk)
# ==========================================
print("=== CLASSIFICATION TASK ===")
y_class = df['high_risk']

# Stratify ensures the 82/17 class imbalance is maintained in both train and test sets
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Model 1A: Logistic Regression (Baseline)
# class_weight='balanced' helps penalize mistakes on the minority (high_risk) class
clf_log = LogisticRegression(class_weight='balanced', random_state=42)
clf_log.fit(X_train_c, y_train_c)
y_pred_log = clf_log.predict(X_test_c)
y_prob_log = clf_log.predict_proba(X_test_c)[:, 1]

print("Logistic Regression Metrics:")
print(f"Accuracy : {accuracy_score(y_test_c, y_pred_log):.4f}")
print(f"Precision: {precision_score(y_test_c, y_pred_log):.4f}")
print(f"Recall   : {recall_score(y_test_c, y_pred_log):.4f}")
print(f"F1-Score : {f1_score(y_test_c, y_pred_log):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test_c, y_prob_log):.4f}\n")

# Model 1B: Random Forest Classifier (Ensemble)
clf_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf_rf.fit(X_train_c, y_train_c)
y_pred_rf = clf_rf.predict(X_test_c)
y_prob_rf = clf_rf.predict_proba(X_test_c)[:, 1]

print("Random Forest Classifier Metrics:")
print(f"Accuracy : {accuracy_score(y_test_c, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test_c, y_pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test_c, y_pred_rf):.4f}")
print(f"F1-Score : {f1_score(y_test_c, y_pred_rf):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test_c, y_prob_rf):.4f}\n")


# ==========================================
# 3. REGRESSION TASK (Target: traffic_volume)
# ==========================================
print("=== REGRESSION TASK ===")
y_reg = df['traffic_volume']

# No stratify parameter needed for continuous regression targets
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Model 2A: Linear Regression (Baseline)
reg_lin = LinearRegression()
reg_lin.fit(X_train_r, y_train_r)
y_pred_lin = reg_lin.predict(X_test_r)

print("Linear Regression Metrics:")
print(f"MAE      : {mean_absolute_error(y_test_r, y_pred_lin):.2f}")
print(f"R-squared: {r2_score(y_test_r, y_pred_lin):.4f}\n")

# Model 2B: Random Forest Regressor (Ensemble)
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)
reg_rf.fit(X_train_r, y_train_r)
y_pred_rf_reg = reg_rf.predict(X_test_r)

print("Random Forest Regressor Metrics:")
print(f"MAE      : {mean_absolute_error(y_test_r, y_pred_rf_reg):.2f}")
print(f"R-squared: {r2_score(y_test_r, y_pred_rf_reg):.4f}")

=== CLASSIFICATION TASK ===
Logistic Regression Metrics:
Accuracy : 0.9251
Precision: 0.7206
Recall   : 0.9276
F1-Score : 0.8111
ROC AUC  : 0.9693

Random Forest Classifier Metrics:
Accuracy : 0.9844
Precision: 0.9363
Recall   : 0.9767
F1-Score : 0.9561
ROC AUC  : 0.9976

=== REGRESSION TASK ===
Linear Regression Metrics:
MAE      : 835.03
R-squared: 0.7086

Random Forest Regressor Metrics:
MAE      : 264.03
R-squared: 0.9464
